In [ ]:
from datasets import load_dataset

# Load the "full" configuration and split
dataset = load_dataset(
    "LeMaterial/LeMat-Synth-Papers",
    "full",  # configuration/subset
    split=None,  # other splits include 'arxiv', 'omg24', 'chemrxiv'
    token=True,
)

# Print column names
print(dataset.column_names)

In [ ]:
# Query function, takes in the split name, which column to check and a list of keywords to return the IDs of the papers that match
def query_db(text_column, split_name, list_keywords):
    ds = dataset[split_name]

    results = {}  # store keyword and corresponding IDs

    for keyword in list_keywords:
        # define filter function for this keyword
        def contains_keyword(example, kw=keyword):
            return (
                example[text_column] is not None
                and kw.lower() in example[text_column].lower()
            )

        # filter dataset
        filtered = ds.filter(contains_keyword)

        # extract IDs
        ids = filtered["id"]

        # store in dictionary
        results[keyword] = ids

    # Print results
    for kw, ids in results.items():
        print(f"\nKeyword: {kw}")
        print(f"Found {len(ids)} entries")
        print(ids)

    return results

In [ ]:
# From a given subset of papers, removes redundancies
def return_nonredundant_ids(results):
    all_ids = []

    for kw, ids in results.items():
        all_ids.extend(ids)

    # Convert to a set to remove duplicates
    unique_ids = set(all_ids)

    # Count them
    print("Number of papers containing at least one keyword:", len(unique_ids))

    return unique_ids

In [ ]:
import pickle

split_name_list = ["arxiv", "omg24", "chemrxiv"]
text_column = "abstract"
list_keywords = [
    "heterogeneous catalysis",
    "heterogeneous catalyst",
    "was heated at",
    "was heated under",
    "thermal treatment",
    "was measured at different temperatures",
    "activation energy",
    "variations",
    "conversion",
    "efficiency",
    "activation energy",
]
"""list_keywords = ["catalyst", "catalysis", "catalytic", "TOF", "activation energy"]"""
db = {}

for split_name in split_name_list:
    result = query_db(
        split_name=split_name,
        text_column=text_column,
        list_keywords=list_keywords,
    )
    nonredundant_ids = return_nonredundant_ids(result)
    print(split_name, len(nonredundant_ids), nonredundant_ids)

    db[split_name] = nonredundant_ids


# export db to pickle file
with open("../results/db_thermocatalysis.pkl", "wb") as f:
    pickle.dump(db, f)

print("Saved db.pkl with keys:", list(db.keys()))

In [ ]:
db

In [ ]:
db.keys()

In [ ]:
db["arxiv"]

In [ ]:
[len(db[key]) for key in db.keys()]

In [ ]:
dataset["arxiv"].to_pandas()

In [ ]:
# print all entries of dataset['arxiv'].to_pandas() where the row "id" is in db['arxiv']
df = dataset["arxiv"].to_pandas()
filtered_df = df[df["id"].isin(db["arxiv"])]
filtered_df

In [ ]:
# print all entries of dataset['arxiv'].to_pandas() where the row "id" is in db['arxiv']
df = dataset["chemrxiv"].to_pandas()
filtered_df_chemrxiv = df[df["id"].isin(db["chemrxiv"])]
filtered_df_chemrxiv

In [ ]:
# print all entries of dataset['arxiv'].to_pandas() where the row "id" is in db['arxiv']
df = dataset["omg24"].to_pandas()
filtered_df_omg24 = df[df["id"].isin(db["omg24"])]
filtered_df_omg24

In [ ]:
import pandas as pd

# concatenate all filtered dataframes
filtered_df_all = pd.concat(
    [filtered_df, filtered_df_chemrxiv, filtered_df_omg24]
)
filtered_df_all

In [ ]:
# sample 1000 random rows from filtered_df_all
filtered_df_all_sampled = filtered_df_all.sample(n=100)
filtered_df_all_sampled

In [ ]:
# download all papers in filtered_df_all_sampled['pdf_url']
import requests

for url in filtered_df_all_sampled["pdf_url"]:
    try:
        response = requests.get(url)
        with open(f"./papers/{url.split('/')[-1]}.pdf", "wb") as f:
            f.write(response.content)
    except Exception as e:
        print(f"Error downloading {url}: {e}")

In [ ]:
import requests

# Setup session with proper headers
session = requests.Session()
session.headers.update(
    {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/pdf,text/html,application/xhtml+xml",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://pubs.rsc.org/",
    }
)

for i, url in enumerate(filtered_df_all_sampled["pdf_url"]):
    try:
        print(f"Downloading {i + 1}/{len(filtered_df_all_sampled)}: {url}")

        response = session.get(url, timeout=30, allow_redirects=True)
        response.raise_for_status()

        with open(f"./papers/{url.split('/')[-1]}.pdf", "wb") as f:
            f.write(response.content)

        # IMPORTANT: Rate limiting for RSC
        # time.sleep(1)  # 2-3 seconds between requests

    except Exception as e:
        print(f"Error downloading {url}: {e}")

print("Download complete!")

In [ ]:
from huggingface_hub import whoami

try:
    user_info = whoami()
    print(f"Logged in as: {user_info['name']}")
except Exception as e:
    print(f"Not logged in: {e}")

In [ ]:
# Load the "full" configuration and split
dataset = load_dataset(
    "amayuelas/LeMat-Synth-Papers-Catalysis-v2",
    name="default",  # or try without this parameter
    token=True,
)

In [ ]:
alfonso_df = dataset["train"].to_pandas()

In [ ]:
import matplotlib.pyplot as plt

df1 = filtered_df_all
df2 = alfonso_df
ids_df1 = set(df1["id"])
ids_df2 = set(df2["id"])

# Get the actual IDs (not just counts)
only_df1 = ids_df1 - ids_df2
only_df2 = ids_df2 - ids_df1
both = ids_df1 & ids_df2

# Create new dataframes for each category
df1_only = df1[df1["id"].isin(only_df1)]
df2_only = df2[df2["id"].isin(only_df2)]
df_both = df1[df1["id"].isin(both)]

print(f"\nRows only in DF1: {len(df1_only)}")
print(f"Rows only in DF2: {len(df2_only)}")
print(f"Rows in both: {len(df_both)}")

categories = ["Only DF1", "Both", "Only DF2"]
counts = [len(only_df1), len(both), len(only_df2)]

plt.figure(figsize=(10, 6))
plt.bar(categories, counts, color=["#ff7f0e", "#2ca02c", "#1f77b4"])
plt.ylabel("Number of IDs")
plt.title("ID Distribution Across DataFrames")
for i, v in enumerate(counts):
    plt.text(
        i, v + max(counts) * 0.02, str(v), ha="center", va="bottom", fontsize=12
    )
plt.show()

In [ ]:
df1_only

In [ ]:
# may need to run uv pip install matplotlib-venn
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

# Use the venn2 function
venn2(
    subsets=(len(df1_only), len(df2_only), len(df_both)),
    set_labels=("keywords", "LLM curated", "overlap"),
)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

# Get unique sources
sources = (
    df1["source"].unique()
    if "source" in df1.columns
    else df2["source"].unique()
)

# Create a figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, source in enumerate(["arxiv", "chemrxiv", "omg24"]):
    # Filter both dataframes by source
    df1_source = (
        df1[df1["source"] == source]
        if "source" in df1.columns
        else pd.DataFrame()
    )
    df2_source = (
        df2[df2["source"] == source]
        if "source" in df2.columns
        else pd.DataFrame()
    )

    # Get sets of IDs for this source
    ids_df1_source = set(df1_source["id"]) if len(df1_source) > 0 else set()
    ids_df2_source = set(df2_source["id"]) if len(df2_source) > 0 else set()

    # Calculate overlaps
    only_df1 = ids_df1_source - ids_df2_source
    only_df2 = ids_df2_source - ids_df1_source
    both = ids_df1_source & ids_df2_source

    # Create Venn diagram
    plt.sca(axes[idx])
    venn2(
        subsets=(len(only_df1), len(only_df2), len(both)),
        set_labels=("keywords", "LLM curated"),
    )
    axes[idx].set_title(
        f"{source.upper()}\n(Total: {len(ids_df1_source | ids_df2_source)})",
        fontsize=14,
        fontweight="bold",
    )

    # Print stats
    print(f"\n{source.upper()}:")
    print(f"  Only in keywords: {len(only_df1)}")
    print(f"  Only in LLM curated: {len(only_df2)}")
    print(f"  In both: {len(both)}")

plt.tight_layout()
plt.show()

In [ ]:
df_both

In [ ]:
import requests

# Setup session with proper headers
session = requests.Session()
session.headers.update(
    {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/pdf,text/html,application/xhtml+xml",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://pubs.rsc.org/",
    }
)

for i, url in enumerate(df_both["pdf_url"]):
    try:
        print(f"Downloading {i + 1}/{len(df_both)}: {url}")

        response = session.get(url, timeout=30, allow_redirects=True)
        response.raise_for_status()

        with open(f"./papers_both/{url.split('/')[-1]}.pdf", "wb") as f:
            f.write(response.content)

        # IMPORTANT: Rate limiting for RSC
        # time.sleep(1)  # 2-3 seconds between requests

    except Exception as e:
        print(f"Error downloading {url}: {e}")

print("Download complete!")

In [ ]:
import re
import time
from pathlib import Path
from urllib.parse import unquote, urlparse

import requests

# Create papers directory
Path("./papers").mkdir(exist_ok=True)

# Setup session with realistic browser headers
session = requests.Session()
session.headers.update(
    {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "none",
    }
)


def get_filename(url, response, index):
    """Extract proper filename from URL or response headers"""

    # Try to get filename from Content-Disposition header
    if "Content-Disposition" in response.headers:
        cd = response.headers["Content-Disposition"]
        filenames = re.findall('filename="?([^"]+)"?', cd)
        if filenames:
            return filenames[0]

    # Parse URL to get filename without query parameters
    parsed = urlparse(url)
    path = unquote(parsed.path)  # Decode URL encoding
    filename = path.split("/")[-1]

    # Remove query parameters if they leaked through
    filename = filename.split("?")[0]

    # If no extension or bad filename, generate one
    if not filename.endswith(".pdf") or len(filename) < 5:
        filename = f"paper_{index:04d}.pdf"

    return filename


failed_downloads = []

for i, url in enumerate(filtered_df_all_sampled["pdf_url"]):
    try:
        print(f"Downloading {i + 1}/{len(filtered_df_all_sampled)}: {url}")

        # Add Referer for the specific domain
        headers = {}
        if "mdpi.com" in url:
            headers["Referer"] = "https://www.mdpi.com/"
        elif "pubs.rsc.org" in url:
            headers["Referer"] = "https://pubs.rsc.org/"

        response = session.get(
            url, timeout=30, allow_redirects=True, headers=headers
        )
        response.raise_for_status()

        # Get proper filename
        filename = get_filename(url, response, i)
        filepath = f"./papers/{filename}"

        # Save file
        with open(filepath, "wb") as f:
            f.write(response.content)

        print(f"  ✓ Saved as: {filename}")

        # Rate limiting
        time.sleep(2)

    except requests.exceptions.HTTPError as e:
        print(f"  ✗ HTTP Error {e.response.status_code}: {url}")
        failed_downloads.append((url, str(e)))
    except Exception as e:
        print(f"  ✗ Error: {url}: {e}")
        failed_downloads.append((url, str(e)))

# Summary
print(f"\n{'=' * 60}")
print(
    f"Successfully downloaded: {len(filtered_df_all_sampled) - len(failed_downloads)}/{len(filtered_df_all_sampled)}"
)
print(f"Failed: {len(failed_downloads)}")

if failed_downloads:
    print("\nFailed downloads:")
    for url, error in failed_downloads:
        print(f"  {url}")
        print(f"    → {error}")